# 序列逆置 （加注意力的seq2seq）
使用attentive sequence to sequence 模型将一个字符串序列逆置。例如 `OIMESIQFIQ` 逆置成 `QIFQISEMIO`(下图来自网络，是一个加attentino的sequence to sequence 模型示意图)
![attentive seq2seq](./seq2seq-attn.jpg)

In [14]:
import numpy as np
import tensorflow as tf
import collections
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import layers, optimizers, datasets
import os,sys,tqdm

## 玩具序列数据生成
生成只包含[A-Z]的字符串，并且将encoder输入以及decoder输入以及decoder输出准备好（转成index）

In [15]:
import random
import string

def randomString(stringLength):
    """Generate a random string with the combination of lowercase and uppercase letters """

    letters = string.ascii_uppercase
    return ''.join(random.choice(letters) for i in range(stringLength))

def get_batch(batch_size, length):
    batched_examples = [randomString(length) for i in range(batch_size)]
    enc_x = [[ord(ch)-ord('A')+1 for ch in list(exp)] for exp in batched_examples]
    y = [[o for o in reversed(e_idx)] for e_idx in enc_x]
    dec_x = [[0]+e_idx[:-1] for e_idx in y]
    return (batched_examples, tf.constant(enc_x, dtype=tf.int32), 
            tf.constant(dec_x, dtype=tf.int32), tf.constant(y, dtype=tf.int32))
print(get_batch(2, 10))

(['GCCWGGITPO', 'BYFAYXDBNN'], <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[ 7,  3,  3, 23,  7,  7,  9, 20, 16, 15],
       [ 2, 25,  6,  1, 25, 24,  4,  2, 14, 14]])>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[ 0, 15, 16, 20,  9,  7,  7, 23,  3,  3],
       [ 0, 14, 14,  2,  4, 24, 25,  1,  6, 25]])>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[15, 16, 20,  9,  7,  7, 23,  3,  3,  7],
       [14, 14,  2,  4, 24, 25,  1,  6, 25,  2]])>)


# 建立sequence to sequence 模型

完成两空，模型搭建以及单步解码逻辑

In [16]:
class mySeq2SeqModel(keras.Model):
    def __init__(self):
        super(mySeq2SeqModel, self).__init__()
        self.v_sz=27
        self.hidden = 128
        self.embed_layer = tf.keras.layers.Embedding(self.v_sz, 64)
        
        self.encoder_cell = tf.keras.layers.SimpleRNNCell(self.hidden)
        self.decoder_cell = tf.keras.layers.SimpleRNNCell(self.hidden)
        
        self.encoder = tf.keras.layers.RNN(self.encoder_cell, 
                                           return_sequences=True, return_state=True)
        self.decoder = tf.keras.layers.RNN(self.decoder_cell, 
                                           return_sequences=True, return_state=True)
        self.dense_attn = tf.keras.layers.Dense(self.hidden)
        self.dense = tf.keras.layers.Dense(self.v_sz)
        
        
    @tf.function
    def call(self, enc_ids, dec_ids):
        '''
        带注意力机制的 seq2seq 前向（训练时整段解码）：
        1. 编码器获得 `enc_out` (batch, enc_len, hidden) 和 `enc_state`；
        2. 解码器对 `dec_ids` 进行并行解码得到 `dec_out` (batch, dec_len, hidden)；
        3. 计算点积注意力 scores = dec_out @ enc_out^T -> (batch, dec_len, enc_len)，softmax 后得到权重；
        4. 用权重加权得到 context (batch, dec_len, hidden)，与 dec_out 拼接并过 `dense_attn`，最后过 `dense` 得到词表 logits。
        '''
        enc_emb = self.embed_layer(enc_ids)  # (b, enc_len, emb)
        enc_out, enc_state = self.encoder(enc_emb)  # enc_out: (b, enc_len, hidden)

        dec_emb = self.embed_layer(dec_ids)  # (b, dec_len, emb)
        dec_out, _ = self.decoder(dec_emb, initial_state=enc_state)  # dec_out: (b, dec_len, hidden)

        # 点积注意力：scores shape (b, dec_len, enc_len)
        scores = tf.matmul(dec_out, enc_out, transpose_b=True)
        attn_weights = tf.nn.softmax(scores, axis=-1)  # 在 enc_len 上归一化

        # context: (b, dec_len, hidden)
        context = tf.matmul(attn_weights, enc_out)

        # 拼接 dec_out 和 context，映射到 hidden，再到词表
        concat = tf.concat([dec_out, context], axis=-1)  # (b, dec_len, hidden*2)
        attn_hidden = self.dense_attn(concat)  # (b, dec_len, hidden)
        logits = self.dense(attn_hidden)  # (b, dec_len, v_sz)
        return logits
    
    
    @tf.function
    def encode(self, enc_ids):
        enc_emb = self.embed_layer(enc_ids) # shape(b_sz, len, emb_sz)
        enc_out, enc_state = self.encoder(enc_emb)
        return enc_out, [enc_out[:, -1, :], enc_state]
    
    def get_next_token(self, x, state, enc_out):
        '''
        单步解码（预测阶段使用）：
        - x: tensor shape (b_sz,) 当前输入 token id
        - state: [prev_context, rnn_state]
        - enc_out: encoder 输出 (b, enc_len, hidden)

        返回: out(token ids), new_state
        '''
        # 嵌入并运行 decoder cell
        inp_emb = self.embed_layer(x)  # (b, emb)
        rnn_state = state[1]
        h, new_rnn_state = self.decoder_cell.call(inp_emb, rnn_state)  # (b, hidden)

        # 计算当前 step 与 encoder 各位置的相似度（点积）
        # scores: (b, enc_len)
        scores = tf.squeeze(tf.matmul(enc_out, tf.expand_dims(h, -1)), axis=-1)
        weights = tf.nn.softmax(scores, axis=-1)  # (b, enc_len)

        # 计算 context: (b, hidden)
        context = tf.reduce_sum(enc_out * tf.expand_dims(weights, -1), axis=1)

        # 合并 context 与当前隐藏，映射到 vocab
        concat = tf.concat([h, context], axis=-1)  # (b, hidden*2)
        attn_hidden = self.dense_attn(concat)  # (b, hidden)
        logits = self.dense(attn_hidden)  # (b, v_sz)
        out = tf.argmax(logits, axis=-1, output_type=tf.int32)

        # new state 保持与 encode 返回结构一致: [context, new_rnn_state]
        return out, [context, new_rnn_state]

# Loss函数以及训练逻辑

In [17]:
@tf.function
def compute_loss(logits, labels):
    losses = tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels)
    losses = tf.reduce_mean(losses)
    return losses

def train_one_step(model, optimizer, enc_x, dec_x, y):
    with tf.GradientTape() as tape:
        logits = model(enc_x, dec_x)
        loss = compute_loss(logits, y)

    # compute gradient
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

def train(model, optimizer, seqlen):
    loss = 0.0
    accuracy = 0.0
    for step in range(2000):
        batched_examples, enc_x, dec_x, y = get_batch(32, seqlen)
        loss = train_one_step(model, optimizer, enc_x, dec_x, y)
        if step % 500 == 0:
            print('step', step, ': loss', loss.numpy())
    return loss

# 训练迭代

In [18]:
optimizer = optimizers.Adam(0.0005)
model = mySeq2SeqModel()
# Warmup call: build model variables before entering tf.function training step.
_dummy_enc = tf.zeros((1, 20), dtype=tf.int32)
_dummy_dec = tf.zeros((1, 20), dtype=tf.int32)
_ = model(_dummy_enc, _dummy_dec)
train(model, optimizer, seqlen=20)

step 0 : loss 3.306928
step 500 : loss 0.8633028
step 1000 : loss 0.27620155
step 1500 : loss 0.15660119


<tf.Tensor: shape=(), dtype=float32, numpy=0.09118388593196869>

# 测试模型逆置能力
首先要先对输入的一个字符串进行encode，然后在用decoder解码出逆置的字符串

测试阶段跟训练阶段的区别在于，在训练的时候decoder的输入是给定的，而在预测的时候我们需要一步步生成下一步的decoder的输入

In [19]:
def sequence_reversal():
    def decode(init_state, steps, enc_out):
        b_sz = tf.shape(init_state[0])[0]
        cur_token = tf.zeros(shape=[b_sz], dtype=tf.int32)
        state = init_state
        collect = []
        for i in range(steps):
            cur_token, state = model.get_next_token(cur_token, state, enc_out)
            collect.append(tf.expand_dims(cur_token, axis=-1))
        out = tf.concat(collect, axis=-1).numpy()
        out = [''.join([chr(idx+ord('A')-1) for idx in exp]) for exp in out]
        return out
    
    batched_examples, enc_x, _, _ = get_batch(32, 20)
    enc_out, state = model.encode(enc_x)
    return decode(state, enc_x.get_shape()[-1], enc_out), batched_examples

def is_reverse(seq, rev_seq):
    rev_seq_rev = ''.join([i for i in reversed(list(rev_seq))])
    if seq == rev_seq_rev:
        return True
    else:
        return False
print([is_reverse(*item) for item in list(zip(*sequence_reversal()))])
print(list(zip(*sequence_reversal())))

[True, True, False, True, True, False, True, False, True, False, True, False, True, True, True, True, False, True, True, True, True, True, True, True, False, False, True, True, True, True, True, True]
[('RZARGGQARUUCYNOEZVDT', 'TDVZEONYCUURAQGGRAZR'), ('IUMEWPJMQWMHWMPTWJSA', 'ASJWTPMWHMWQMJPWEMUI'), ('NUXPITNYLXPHZWWVRWHY', 'YHWRVWWZHPXLYNTIPXUN'), ('QAMRNGLFFEXXIUTXQDIG', 'GIDQXTUIXXEFFLGNRMAQ'), ('CVTFRHVFUIOPKDVTRUVY', 'EVUOTVDKPOIUFVHRFTVC'), ('BLPOGTXOKDDEYYRUFKCZ', 'ZCKFURYYEDDKOXTGOPLB'), ('FDTMOEDCEDXLBVPJVUXZ', 'ZUAPIHVFLHSEUSGOMTDF'), ('ZCRQQYROZDOUMJKPACAC', 'CUCAPKJMUODZORYQQRCZ'), ('WQRBQVUYDCUYGNBKFGCN', 'NCGFKBNGYUCDYUVQBRQW'), ('FXFUASJVAUXSJDWZSLRD', 'DRLSZWDJSXUAVJSAUFXF'), ('BJVPTAUOGQTYQUCSCRNB', 'BNRVSCMQYTQGOUATPVJB'), ('DMXCEPIUJTGOSCJLHDLY', 'YLDHLJCSOGTJUIPECXMD'), ('ZAPNKZFDOSCGOMPDVVYL', 'LYVVDPMOGCSODFZKNPAZ'), ('NXIUIKXVEFFPDTBRWQJO', 'OJQWRBTDPFFEVXKIUIXN'), ('RKMSSSODYPIFAAWDATYC', 'CYTADWAAFIPYDOSSSMKR'), ('XXZGMUNKJPBKKJAPUDJK', 'KJDUPAJKKBPJKNUMGZXX')